# 02. NumPy: 벡터화, broadcasting, 수치 안정성, 선형대수

딥러닝 프레임워크의 핵심 연산은 NumPy와 같은 배열 사고방식에서 출발합니다.
이 노트북에서는 shape·axis·dtype을 추적하면서 **벡터화, broadcasting, 난수 생성기,
안정적인 softmax/cross-entropy, 행렬곱, cosine similarity, 최소제곱, SVD**를 연습합니다.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    """현재 폴더부터 위로 올라가며 프로젝트 루트를 찾습니다."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "pyproject.toml과 src/가 있는 프로젝트 폴더를 찾지 못했습니다."
    )


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("project:", PROJECT_ROOT)
print("python :", sys.executable)

In [ ]:
from time import perf_counter

import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 배열의 기본 계약: shape, dtype, ndim

AI 코드에서 가장 흔한 오류는 값 자체보다 shape의 의미를 잃는 것입니다.
아래 배열은 `(batch, features)` 구조입니다.

In [ ]:
features = np.array(
    [
        [2.0, 0.5, 1.0],
        [1.0, 1.5, 0.0],
        [3.0, 0.2, 1.0],
        [0.5, 2.0, 0.0],
    ],
    dtype=np.float32,
)

assert features.shape == (4, 3)
assert features.ndim == 2
assert features.dtype == np.float32
print("shape:", features.shape, "dtype:", features.dtype)
features

## 2. Python loop와 벡터화

벡터화는 연산 의도를 배열 단위로 표현해 C/BLAS 수준 구현을 활용합니다.
성능 측정은 환경에 따라 달라지므로 속도를 assert하지 않고 결과의 동등성만 검증합니다.

In [ ]:
values = np.linspace(-3, 3, 200_000, dtype=np.float64)

started = perf_counter()
loop_result = np.array([value * value + 2 * value + 1 for value in values])
loop_ms = (perf_counter() - started) * 1_000

started = perf_counter()
vectorized_result = values**2 + 2 * values + 1
vectorized_ms = (perf_counter() - started) * 1_000

assert np.allclose(loop_result, vectorized_result)
print(f"loop={loop_ms:.2f} ms, vectorized={vectorized_ms:.2f} ms")

## 3. Broadcasting

`(batch, features)` 배열에서 `(features,)` 평균을 빼면 NumPy가 마지막 축을 맞춰 확장합니다.
실제 복사가 아니라 호환 가능한 shape 규칙이 적용됩니다.

## 4. `axis`와 `keepdims`

`axis=0`은 batch를 모으고 feature별 통계를, `axis=1`은 feature를 모으고 샘플별 통계를 냅니다.
`keepdims=True`는 축을 길이 1로 유지해 후속 broadcasting을 명확하게 합니다.

In [ ]:
feature_mean = features.mean(axis=0)  # (features,)
centered = features - feature_mean  # (batch, features) - (features,)

assert feature_mean.shape == (3,)
assert centered.shape == features.shape
assert np.allclose(centered.mean(axis=0), 0.0, atol=1e-6)
centered

In [ ]:
row_sums = features.sum(axis=1, keepdims=True)
normalized_rows = features / np.where(row_sums == 0, 1, row_sums)

assert row_sums.shape == (4, 1)
assert np.allclose(normalized_rows.sum(axis=1), 1.0)
normalized_rows

## 5. Boolean mask와 fancy indexing

조건을 배열로 만들고 필요한 샘플만 선택합니다. 원본 순서를 보존하지만 대부분 새 배열을 만듭니다.

## 6. 전역 난수 대신 `Generator` 사용

함수 내부에 독립적인 `Generator`를 전달하면 실험 간 난수 상태가 덜 얽힙니다.
동일한 seed로 만든 두 generator는 같은 수열을 생성합니다.

In [ ]:
confidence = np.array([0.92, 0.41, 0.77, 0.33])
review_mask = confidence < 0.6
review_rows = features[review_mask]

assert review_mask.tolist() == [False, True, False, True]
assert review_rows.shape == (2, 3)
review_rows

In [ ]:
rng_a = np.random.default_rng(42)
rng_b = np.random.default_rng(42)

sample_a = rng_a.normal(loc=0.0, scale=1.0, size=(2, 4))
sample_b = rng_b.normal(loc=0.0, scale=1.0, size=(2, 4))

assert np.array_equal(sample_a, sample_b)
sample_a

## 7. 수치적으로 안정적인 softmax

`exp(1000)`은 overflow합니다. 각 행의 최댓값을 먼저 빼도 softmax 결과는 같고 지수 범위는 안전해집니다.

In [ ]:
def stable_softmax(logits: np.ndarray, axis: int = -1) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    shifted = logits - logits.max(axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)


large_logits = np.array([[1_000.0, 1_001.0, 999.0], [-1_000.0, -999.0, -998.0]])
probabilities = stable_softmax(large_logits)

assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1.0)
probabilities

## 8. log-sum-exp와 cross-entropy

정답 클래스의 `-log(probability)`가 cross-entropy입니다. 확률을 먼저 계산한 뒤 log를 취하는 대신
logits 공간의 log-sum-exp를 사용하면 극단적인 값에서도 안정적입니다.

In [ ]:
def logsumexp(values: np.ndarray, axis: int = -1) -> np.ndarray:
    maximum = values.max(axis=axis, keepdims=True)
    return (
        maximum + np.log(np.exp(values - maximum).sum(axis=axis, keepdims=True))
    ).squeeze(axis)


def cross_entropy_from_logits(logits: np.ndarray, targets: np.ndarray) -> float:
    row_indices = np.arange(len(logits))
    losses = logsumexp(logits, axis=1) - logits[row_indices, targets]
    return float(losses.mean())


targets = np.array([1, 2])
loss = cross_entropy_from_logits(large_logits, targets)
expected = -np.log(probabilities[np.arange(2), targets]).mean()
assert np.isclose(loss, expected)
print("cross-entropy:", round(loss, 6))

## 9. 행렬곱: batch 입력과 선형 계층

선형 계층은 `X @ W + b`입니다. `(batch, input_dim) @ (input_dim, output_dim)`의 결과는
`(batch, output_dim)`이 됩니다.

## 10. Cosine similarity

임베딩 검색은 방향이 비슷한 벡터를 찾습니다. 0 벡터의 norm은 작은 epsilon으로 보호합니다.

In [ ]:
rng = np.random.default_rng(7)
weights = rng.normal(0, 0.1, size=(3, 2))
bias = np.zeros(2)
logits = features @ weights + bias

assert logits.shape == (4, 2)
logits

In [ ]:
def cosine_similarity(
    query: np.ndarray, documents: np.ndarray, eps: float = 1e-12
) -> np.ndarray:
    query = np.asarray(query, dtype=float)
    documents = np.asarray(documents, dtype=float)
    query_norm = max(float(np.linalg.norm(query)), eps)
    document_norms = np.maximum(np.linalg.norm(documents, axis=1), eps)
    return (documents @ query) / (document_norms * query_norm)


query = np.array([1.0, 0.0, 1.0])
similarities = cosine_similarity(query, features)
assert similarities.shape == (4,)
assert np.all((-1 <= similarities) & (similarities <= 1))
similarities

## 11. 최소제곱으로 선형 회귀 풀기

역행렬을 직접 계산하기보다 `np.linalg.lstsq`를 사용합니다. rank가 부족한 경우에도 더 안정적입니다.

## 12. SVD와 저차원 표현

중심화한 행렬을 SVD로 분해하면 주성분 방향을 얻을 수 있습니다.
여기서는 첫 두 방향으로 투영하고 shape과 복원 오차를 확인합니다.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y = 1.5 * x + 0.7
design = np.column_stack([x, np.ones_like(x)])

coefficients, residuals, rank, singular_values = np.linalg.lstsq(design, y, rcond=None)
slope, intercept = coefficients

assert np.isclose(slope, 1.5)
assert np.isclose(intercept, 0.7)
print({"slope": slope, "intercept": intercept, "rank": rank})

In [ ]:
centered64 = features.astype(np.float64) - features.mean(axis=0)
u, singular_values, vt = np.linalg.svd(centered64, full_matrices=False)
projected_2d = centered64 @ vt[:2].T
reconstructed = projected_2d @ vt[:2]

assert projected_2d.shape == (4, 2)
assert np.linalg.norm(centered64 - reconstructed) >= 0
print("singular values:", singular_values)
projected_2d

## 13. View와 copy

slice는 원본 메모리를 공유할 수 있습니다. 전처리 중 원본을 보존해야 한다면 `.copy()`를 명시하세요.

In [ ]:
original = np.arange(8)
view = original[2:5]
independent = original[2:5].copy()

view[0] = 999
independent[1] = -1

assert original[2] == 999  # view 수정이 원본에 반영됨
assert original[3] != -1  # copy 수정은 원본과 독립적
original, view, independent

## TODO 1: feature 표준화 함수

구현을 지운 뒤 `mean=0`, `std=1`이 되도록 다시 작성하세요. 분산이 0인 열은 0으로 유지해야 합니다.

In [ ]:
def standardize(matrix: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    # TODO: 열별 평균과 표준편차를 사용해 표준화하세요.
    matrix = np.asarray(matrix, dtype=np.float64)
    mean = matrix.mean(axis=0, keepdims=True)
    std = matrix.std(axis=0, keepdims=True)
    safe_std = np.where(std < eps, 1.0, std)
    return (matrix - mean) / safe_std


standardized = standardize(features)
assert np.allclose(standardized.mean(axis=0), 0.0, atol=1e-7)
assert np.allclose(standardized.std(axis=0), 1.0, atol=1e-7)
standardized

## TODO 2: batch query-document 유사도

loop 없이 `(queries, dim) @ (documents, dim).T`를 사용해 모든 조합의 cosine similarity를 계산하세요.

In [ ]:
def pairwise_cosine(
    queries: np.ndarray, documents: np.ndarray, eps: float = 1e-12
) -> np.ndarray:
    # TODO: 각 행을 L2 정규화한 뒤 행렬곱하세요.
    query_norms = np.maximum(np.linalg.norm(queries, axis=1, keepdims=True), eps)
    document_norms = np.maximum(np.linalg.norm(documents, axis=1, keepdims=True), eps)
    return (queries / query_norms) @ (documents / document_norms).T


pairwise = pairwise_cosine(features[:2], features)
assert pairwise.shape == (2, 4)
assert np.allclose(np.diag(pairwise[:, :2]), 1.0)
pairwise

## 정리

- 항상 `(batch, sequence, hidden)`처럼 각 축의 의미를 말로 적습니다.
- 벡터화는 속도뿐 아니라 코드의 수학적 의도를 선명하게 합니다.
- softmax/log 연산은 overflow와 underflow를 고려합니다.
- 역행렬보다 `solve`, `lstsq`, SVD 같은 안정적인 도구를 우선합니다.
- 다음 노트북에서는 같은 원칙을 표 형태 데이터와 데이터 누수 문제에 적용합니다.